# BIO span tagger

The `+ BIO output` row of Table 1. Instead of one logit per token, three classes
— Outside / Begin / Inside — so the model has to learn where a span *starts*,
not only which tokens look suspicious.

This is the only change in the paper that alters the output rather than the
input. Representation-shift and caption features are both fused in, 45
dimensions in total.

Colab, one T4.

## Environment

In [ ]:
!nvidia-smi


Mon Jul 27 21:32:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Features

Representation-shift and NLI features, extracted earlier and restored from
Drive.

In [ ]:
!find /content -name "*.npz" -not -path "*/drive/*" | head -20
print("---")
!ls /content/work | head -20


find: ‘/content/drive/MyDrive/large_files/large_files/output’: Input/output error
/content/work/feats/hidden_feats_test_it.npz
/content/work/feats/hidden_feats_test_fr.npz
/content/work/feats/nli_feats_train_fr.npz
/content/work/feats/nli_feats_test_zh.npz
/content/work/feats/nli_feats_test_en.npz
/content/work/feats/nli_feats_test_it.npz
/content/work/feats/hidden_feats_train_en.npz
/content/work/feats/hidden_feats_train_fr.npz
/content/work/feats/hidden_feats_test_zh.npz
/content/work/feats/hidden_feats_train_it.npz
/content/work/feats/nli_feats_train_en.npz
/content/work/feats/nli_feats_train_zh.npz
/content/work/feats/nli_feats_test_fr.npz
/content/work/feats/hidden_feats_test_en.npz
/content/work/feats/hidden_feats_train_zh.npz
/content/work/feats/nli_feats_train_it.npz
/content/work/__MACOSX/feats/._nli_feats_test_zh.npz
/content/work/__MACOSX/feats/._hidden_feats_train_fr.npz
/content/work/__MACOSX/feats/._hidden_feats_test_zh.npz
/content/work/__MACOSX/feats/._nli_feats_train_z

In [ ]:
!mv /content/work/feats/*.npz /content/work/ && rm -rf /content/work/feats /content/work/__MACOSX
!ls /content/work/*.npz | wc -l


16


## Configuration

In [ ]:
!pip install -q transformers==4.* sentencepiece
import os, json, random, zipfile
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import spearmanr

from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = "/content/work"; DISTRIB = f"{OUT_DIR}/distrib"
os.makedirs(OUT_DIR, exist_ok=True)

# data (4 MB) straight from the organizers
!wget -q https://a3s.fi/mickusti-2007780-pub/shroom-visions-data.zip -O /tmp/d.zip
!unzip -oq /tmp/d.zip -d {OUT_DIR}
# features from Drive
with zipfile.ZipFile('/content/feats.zip') as z:
    z.extractall(OUT_DIR)
print(sorted(os.listdir(DISTRIB))[:3], "|", len([f for f in os.listdir(OUT_DIR) if f.endswith('.npz')]), "npz")

MODEL_ID   = "xlm-roberta-large"
LANGS      = ["en", "fr", "it", "zh"]
CATEGORIES = ["invention", "mischaracterization", "OCR", "miscounting", "other"]
MAX_LEN, BATCH, EPOCHS, LR, SEED = 256, 8, 5, 1e-5, 13
VIS_DIM, NLI_DIM, FEAT_PROJ = 38, 7, 64
USE_VIS = USE_NLI = True
FEAT_DIM = VIS_DIM + NLI_DIM
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['shroom-vision.test.en.unlabeled.jsonl', 'shroom-vision.test.fr.unlabeled.jsonl', 'shroom-vision.test.it.unlabeled.jsonl'] | 16 npz
device: cuda


## Scorer, data and features

Unchanged from the text-only notebook. `features_for_tokens` averages each
feature block over the characters an XLM-R token covers, which is how two
different tokenizers are reconciled.

In [ ]:
def score_cor(ref_dict, pred_dict, label_filtered_=None):
    assert ref_dict['id'] == pred_dict['id']
    ref_vec = [0.] * ref_dict['text_len']; pred_vec = [0.] * ref_dict['text_len']
    ref_labels = (ref_dict['labels'] if label_filtered_ is None
                  else [s for s in ref_dict['labels'] if s['label'] == label_filtered_])
    pred_labels = (pred_dict['labels'] if label_filtered_ is None
                   else [s for s in pred_dict['labels'] if s['label'] == label_filtered_])
    for span in ref_labels:
        for idx in range(span['start'], span['end']): ref_vec[idx] += span['prob']
    for span in pred_labels:
        for idx in range(span['start'], span['end']): pred_vec[idx] = span['prob']
    rc = {round(f, 8) for f in ref_vec}; pc = {round(f, 8) for f in pred_vec}
    if len(pc) == 1 or len(rc) == 1:
        if len(pc) != len(rc): return 0.0
        if rc == {0.0}: return float(pc == {0.0})
        return float(pc != {0.0})
    return spearmanr(ref_vec, pred_vec).correlation

def score_cor_lbl(r, p):
    all_labels = {s['label'] for d in [r, p] for s in d['labels']}
    return (sum(score_cor(r, p, l) for l in all_labels) / len(all_labels)
            if all_labels else 1.0)

def score_iou(r, p):
    ri = {i for s in r['labels'] for i in range(s['start'], s['end'])}
    pi = {i for s in p['labels'] for i in range(s['start'], s['end'])}
    return 1. if not pi and not ri else len(ri & pi) / len(ri | pi)

def evaluate(refs, preds):
    refs = sorted(refs, key=lambda r: r['id']); preds = sorted(preds, key=lambda r: r['id'])
    return {'Cor': float(np.mean([score_cor(r, p) for r, p in zip(refs, preds)])),
            'Cor_lbl': float(np.mean([score_cor_lbl(r, p) for r, p in zip(refs, preds)])),
            'IoU': float(np.mean([score_iou(r, p) for r, p in zip(refs, preds)]))}

def load(lang, split_name):
    kind = "labeled" if split_name == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split_name}.{lang}.{kind}.jsonl", encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]

def split_dev(rows, fraction=0.2, seed=SEED):
    clean = [r for r in rows if not r["labels"]]; dirty = [r for r in rows if r["labels"]]
    rng = random.Random(seed); rng.shuffle(clean); rng.shuffle(dirty)
    nc, nd = int(len(clean) * fraction), int(len(dirty) * fraction)
    dev = clean[:nc] + dirty[:nd]; train = clean[nc:] + dirty[nd:]
    rng.shuffle(dev); rng.shuffle(train)
    return train, dev

def char_targets(row):
    n = len(row["response"])
    prob = np.zeros(n, dtype=np.float32); cat = np.full(n, -1, dtype=np.int64)
    for span in row.get("labels", []):
        for i in range(span["start"], min(span["end"], n)):
            if span["prob"] >= prob[i]:
                prob[i] = span["prob"]; cat[i] = CATEGORIES.index(span["label"])
    return prob, cat

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def encode(row):
    enc = tokenizer(row["response"], text_pair=row["prompt"], return_offsets_mapping=True,
                    truncation="only_first", max_length=MAX_LEN)
    seq = enc.sequence_ids()
    keep = [i for i, (a, b) in enumerate(enc["offset_mapping"]) if b > a and seq[i] == 0]
    return enc, keep

def load_char_features(prefix, dim):
    out = {}
    for split in ["train", "test"]:
        for lang in LANGS:
            path = f"{OUT_DIR}/{prefix}_{split}_{lang}.npz"
            if not os.path.exists(path): continue
            d = np.load(path)
            for key in d.files:
                if not key.endswith("|f"): continue
                rid = key[:-2]; offs, feats = d[rid + "|off"], d[key]
                if len(offs) == 0: continue
                arr = np.zeros((int(offs[-1][1]), dim), dtype=np.float32)
                for (a, b), f in zip(offs, feats): arr[a:b] = f
                out[rid] = arr
    return out

VIS = load_char_features("hidden_feats", VIS_DIM)
NLI = load_char_features("nli_feats", NLI_DIM)
print(f"visual: {len(VIS)}   nli: {len(NLI)}")

def _stats(store, dim):
    s = np.concatenate([v for _, v in list(store.items())[:2000]], axis=0)
    return s.mean(0), np.maximum(s.std(0), 1e-2)

VIS_MU, VIS_SD = _stats(VIS, VIS_DIM); NLI_MU, NLI_SD = _stats(NLI, NLI_DIM)

def features_for_tokens(rid, offsets):
    parts, present = [], 1.0
    for store, dim, mu, sd in [(VIS, VIS_DIM, VIS_MU, VIS_SD), (NLI, NLI_DIM, NLI_MU, NLI_SD)]:
        arr = store.get(rid); block = np.zeros((len(offsets), dim), dtype=np.float32)
        if arr is None: present = 0.0
        else:
            for i, (a, b) in enumerate(offsets):
                b = min(b, len(arr))
                if b > a: block[i] = arr[a:b].mean(axis=0)
            block = (block - mu) / sd
        parts.append(block)
    return np.concatenate(parts, axis=1), present


visual: 20000   nli: 19996


## BIO tagger, training and evaluation

Outside dominates roughly 90% of tokens, so the loss is class-weighted; without
it the model predicts all-Outside.

In [ ]:
O, B, I = 0, 1, 2

def bio_targets(offsets, prob_arr):
    out, prev = [], False
    for a, b in offsets:
        p = float(prob_arr[a:b].max()) if b > a else 0.0
        hall = p > 0
        out.append(O if not hall else (I if prev else B))
        prev = hall
    return out

class BioData(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
        self.cache = [encode(r) for r in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]; enc, keep = self.cache[i]
        offs = [enc["offset_mapping"][k] for k in keep]
        feat, present = features_for_tokens(row["id"], offs)
        item = {"input_ids": torch.tensor(enc["input_ids"]),
                "attention_mask": torch.tensor(enc["attention_mask"]),
                "keep": torch.tensor(keep, dtype=torch.long),
                "feat": torch.tensor(feat),
                "present": torch.tensor(present, dtype=torch.float)}
        if self.labeled:
            prob, cat = char_targets(row)
            item["bio"] = torch.tensor(bio_targets(offs, prob), dtype=torch.long)
            tc = []
            for a, b in offs:
                seg = cat[a:b]; seg = seg[seg >= 0]
                tc.append(int(np.bincount(seg).argmax()) if len(seg) else -100)
            item["tok_cat"] = torch.tensor(tc, dtype=torch.long)
        return item

def collate_bio(batch):
    pad = tokenizer.pad_token_id; n = len(batch)
    ml = max(len(b["input_ids"]) for b in batch); mk = max(len(b["keep"]) for b in batch)
    out = {"input_ids": torch.full((n, ml), pad, dtype=torch.long),
           "attention_mask": torch.zeros((n, ml), dtype=torch.long),
           "keep": torch.zeros((n, mk), dtype=torch.long),
           "keep_mask": torch.zeros((n, mk), dtype=torch.bool),
           "feat": torch.zeros((n, mk, FEAT_DIM)),
           "present": torch.stack([b["present"] for b in batch])}
    lab = "bio" in batch[0]
    if lab:
        out["bio"] = torch.full((n, mk), -100, dtype=torch.long)
        out["tok_cat"] = torch.full((n, mk), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L, K = len(b["input_ids"]), len(b["keep"])
        out["input_ids"][i, :L] = b["input_ids"]; out["attention_mask"][i, :L] = b["attention_mask"]
        out["keep"][i, :K] = b["keep"]; out["keep_mask"][i, :K] = True
        out["feat"][i, :K] = b["feat"]
        if lab: out["bio"][i, :K] = b["bio"]; out["tok_cat"][i, :K] = b["tok_cat"]
    return out

class BioTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.feat_proj = nn.Sequential(nn.Linear(FEAT_DIM, FEAT_PROJ), nn.GELU(),
                                       nn.LayerNorm(FEAT_PROJ))
        h += FEAT_PROJ
        self.bio_head = nn.Linear(h, 3)                 # was a single logit
        self.cat_head = nn.Linear(h, len(CATEGORIES))
    def forward(self, input_ids, attention_mask, keep, keep_mask, feat, present):
        hid = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        tok = torch.gather(hid, 1, keep.unsqueeze(-1).expand(-1, -1, hid.size(-1)))
        tok = torch.cat([tok, self.feat_proj(feat) * present.view(-1, 1, 1)], dim=-1)
        tok = self.dropout(tok)
        return self.bio_head(tok), self.cat_head(tok)

def run_epoch_bio(model, loader, opt=None, sched=None, log_every=50):
    train = opt is not None
    model.train() if train else model.eval()
    # O dominates ~90% of tokens, so weight it down or the model predicts all-O
    w = torch.tensor([1.0, 6.0, 4.0], device=DEVICE)
    ce_bio = nn.CrossEntropyLoss(ignore_index=-100, weight=w)
    ce_cat = nn.CrossEntropyLoss(ignore_index=-100)
    total, nb = 0.0, 0
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            bio_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                         batch["keep"], batch["keep_mask"],
                                         batch["feat"], batch["present"])
            l_bio = ce_bio(bio_logit.reshape(-1, 3), batch["bio"].reshape(-1))
            valid = (batch["tok_cat"] != -100).any()
            l_cat = (ce_cat(cat_logit.reshape(-1, len(CATEGORIES)),
                            batch["tok_cat"].reshape(-1)) if valid else bio_logit.sum() * 0.0)
            loss = l_bio + 0.5 * l_cat
        if train:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        total += loss.item(); nb += 1
        if train and step % log_every == 0:
            print(f"  step {step}/{len(loader)}  loss {total/nb:.4f}", flush=True)
    return total / max(nb, 1)

@torch.no_grad()
def predict_bio(model, rows):
    model.eval()
    ds = BioData(rows, labeled=False)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False, collate_fn=collate_bio)
    res, cur = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        bio_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                     batch["keep"], batch["keep_mask"],
                                     batch["feat"], batch["present"])
        p = torch.softmax(bio_logit, -1).cpu().numpy()
        cats = cat_logit.argmax(-1).cpu().numpy()
        for i in range(len(p)):
            row = rows[cur]; enc, keep = ds.cache[cur]; cur += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, np.float32); cc = np.zeros(n, np.int64)
            for j, (a, b) in enumerate(offs):
                cp[a:min(b, n)] = p[i][j][B] + p[i][j][I]   # hallucinated = B or I
                cc[a:min(b, n)] = cats[i][j]
            res.append((cp, cc))
    return res

def to_spans(cp, cc, thr):
    spans, n, i = [], len(cp), 0
    while i < n:
        if cp[i] < thr: i += 1; continue
        j, c = i, cc[i]
        while j < n and cp[j] >= thr and cc[j] == c: j += 1
        spans.append({"start": int(i), "end": int(j),
                      "prob": float(round(float(np.mean(cp[i:j])), 6)),
                      "label": CATEGORIES[int(c)]})
        i = j
    return spans

def build_preds(rows, preds, thr):
    return [{"id": r["id"], "labels": to_spans(cp, cc, thr)} for r, (cp, cc) in zip(rows, preds)]

def tune_threshold(dev_rows, preds):
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])} for r in dev_rows]
    best = (None, -1, None)
    for t in np.arange(0.10, 0.91, 0.05):
        s = evaluate(refs, build_preds(dev_rows, preds, float(t)))
        if s["Cor"] > best[1]: best = (float(t), s["Cor"], s)
    return best

train_rows, dev_rows = [], []
for lang in LANGS:
    tr, dv = split_dev(load(lang, "train")); train_rows += tr; dev_rows += dv
print(f"train={len(train_rows)}  dev={len(dev_rows)}")

model = BioTagger().to(DEVICE)
model.encoder.embeddings.requires_grad_(False)
loader = DataLoader(BioData(train_rows), batch_size=BATCH, shuffle=True, collate_fn=collate_bio)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=LR, weight_decay=0.01, foreach=False)
sched = get_linear_schedule_with_warmup(opt, int(0.1*len(loader)*EPOCHS), len(loader)*EPOCHS)

for ep in range(EPOCHS):
    l = run_epoch_bio(model, loader, opt, sched)
    print(f"epoch {ep+1}: loss {l:.4f}", flush=True)
    torch.save(model.state_dict(), "/content/drive/MyDrive/bio_tagger.pt")

print(f"\n{'lang':<6}{'thr':>6}{'Cor':>9}{'Cor_lbl':>9}{'IoU':>9}")
thresholds = {}
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    cps = predict_bio(model, rows)
    thr, cor, s = tune_threshold(rows, cps); thresholds[lang] = thr
    print(f"{lang:<6}{thr:>6.2f}{s['Cor']:>9.3f}{s['Cor_lbl']:>9.3f}{s['IoU']:>9.3f}")

for lang in LANGS:
    rows = load(lang, "test")
    preds = build_preds(rows, predict_bio(model, rows), thresholds[lang])
    path = f"/content/drive/MyDrive/predictions_bio_{lang}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for p in preds: fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"{lang}: {len(preds)} rows -> {path}")


train=12085  dev=3017


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

  step 50/1511  loss 1.7140
  step 100/1511  loss 1.6739
  step 150/1511  loss 1.6091
  step 200/1511  loss 1.5653
  step 250/1511  loss 1.5315
  step 300/1511  loss 1.4983
  step 350/1511  loss 1.4735
  step 400/1511  loss 1.4599
  step 450/1511  loss 1.4423
  step 500/1511  loss 1.4216
  step 550/1511  loss 1.4138
  step 600/1511  loss 1.3995
  step 650/1511  loss 1.3876
  step 700/1511  loss 1.3764
  step 750/1511  loss 1.3651
  step 800/1511  loss 1.3524
  step 850/1511  loss 1.3422
  step 900/1511  loss 1.3335
  step 950/1511  loss 1.3247
  step 1000/1511  loss 1.3152
  step 1050/1511  loss 1.3125
  step 1100/1511  loss 1.3074
  step 1150/1511  loss 1.2995
  step 1200/1511  loss 1.2929
  step 1250/1511  loss 1.2896
  step 1300/1511  loss 1.2841
  step 1350/1511  loss 1.2800
  step 1400/1511  loss 1.2731
  step 1450/1511  loss 1.2690
  step 1500/1511  loss 1.2661
epoch 1: loss 1.2650
  step 50/1511  loss 1.0825
  step 100/1511  loss 1.1034
  step 150/1511  loss 1.0861
  step 200/15

## Export

In [ ]:
from IPython.display import FileLink
import shutil
!cd /content/drive/MyDrive && zip -q /content/preds_bio.zip predictions_bio_*.jsonl && unzip -l /content/preds_bio.zip
FileLink('/content/preds_bio.zip')


Archive:  /content/preds_bio.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   378545  2026-07-28 00:19   predictions_bio_en.jsonl
   450860  2026-07-28 00:19   predictions_bio_fr.jsonl
   323790  2026-07-28 00:20   predictions_bio_it.jsonl
   150361  2026-07-28 00:21   predictions_bio_zh.jsonl
---------                     -------
  1303556                     4 files


/content/preds_bio.zip